# 05 - Claim analytics

Explore the claims analytics layer on the **synthetic** sample claims (`data/claims/claims_sample.csv`). Costs are indicative assumptions from `config.yaml`, not real insurer data.

**Running on Google Colab?** Upload or clone the project, then run once:

```python
from google.colab import drive
drive.mount("/content/drive")          # optional: keep data and models on Drive
%cd /content/iClaim
%pip install -q -r requirements.txt
```
Use a GPU runtime (Runtime > Change runtime type) for training.

In [ ]:
# --- Setup (works locally and in Google Colab) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.config import load_config, resolve_path, set_seed
cfg = load_config()
set_seed(cfg["project"]["seed"])
print("Project root:", ROOT)

In [ ]:
from src import analytics

claims = analytics.load_claims(cfg, include_log=False)
print(len(claims), 'claims | data source:', claims['data_source'].unique())
claims.head()

## Headline KPIs

In [ ]:
analytics.compute_kpis(claims)

## Damage frequency and severity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
freq = analytics.damage_frequency(claims)
sns.barplot(data=freq, x="damage_class", y="claims", ax=axes[0], color="#2a78d6")
axes[0].set_title("Damage type frequency"); axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=20)

sev = analytics.severity_distribution(claims)
sns.barplot(data=sev, x="severity", y="claims", ax=axes[1], palette=["#a9c8f0", "#5b9be0", "#1f5fb0"])
axes[1].set_title("Severity distribution"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()

## Indicative repair-cost ranges

In [ ]:
analytics.cost_summary(claims)

## Claim-processing priority

Rule (from `config.yaml`): low confidence -> manual review; otherwise priority follows severity.

In [ ]:
pri = analytics.priority_breakdown(claims)
display(pri)
pd.crosstab(claims["severity"], claims["priority"])

## Claims whose amount exceeds the indicative range

In [ ]:
flagged = claims[claims["amount_flag"] == "above_range"]
print(f"{len(flagged)} of {len(claims)} claims flagged for review")
flagged[["claim_id", "damage_class", "severity", "est_cost_high", "claimed_amount"]].head(10)

## Try the priority rule

Change the inputs to see how routing responds.

In [ ]:
for sev, conf in [("severe", 0.92), ("moderate", 0.85), ("minor", 0.95), ("severe", 0.55)]:
    print(f"{sev:9s} conf={conf:.2f} -> {analytics.priority_for(sev, conf, cfg)}")